# Data Cleaning

## Objective

The objective of this phase is to prepare the datasets for analysis by validating data types, handling missing values, investigating duplicate records, and improving overall data quality.

## Data Cleaning Tasks

- Load all datasets
- Validate data types
- Convert date columns to datetime
- Handle missing values
- Investigate duplicate records
- Validate cleaned datasets

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Step 1: Validate Data Types

In this step, the data types of each dataset are validated to ensure that every column has the appropriate type before applying any cleaning operations.

In [2]:
import pandas as pd

from retailpulse.utils import (check_data_types , load_datasets )

In [3]:
(
    orders,
    customers,
    geolocation,
    items,
    payments,
    reviews,
    products,
    sellers,
    categories,
) = load_datasets()

In [4]:
check_data_types(orders, "Orders")
check_data_types(products, "Products")
check_data_types(customers, "Customers")
check_data_types(items, "Order Items")
check_data_types(payments, "Order Payments")
check_data_types(reviews, "Order Reviews")
check_data_types(sellers, "Sellers")
check_data_types(categories, "Product Category Name Translation")
check_data_types(geolocation, "Geolocation")

Dataset: Orders
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object
Dataset: Products
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object
Dataset: Customers
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object
Dataset: Order Items
order_id                   str
order_item_id            int64
product_id    

### Observations

- Most categorical columns are correctly stored as string (`str`).
- Numerical columns are stored using appropriate numeric data types (`int64` and `float64`).
- All date-related columns are still stored as strings and need to be converted to the `datetime` data type.
- No incorrect data types were identified in the remaining columns.

## Step 2: Convert Date Columns

Date and time columns are currently stored as strings (`str`). In this step, these columns will be converted to the `datetime` data type to enable time-based analysis such as extracting years, months, weekdays, and calculating delivery durations.

In [5]:
from retailpulse.utils import convert_to_datetime

In [6]:
orders_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders = convert_to_datetime(orders, orders_date_columns)

In [7]:
items_date_columns = [
    "shipping_limit_date",
]

items = convert_to_datetime(items, items_date_columns)

In [8]:
reviews_date_columns = [
    "review_creation_date",
    "review_answer_timestamp",
]

reviews = convert_to_datetime(reviews, reviews_date_columns)

In [9]:
check_data_types(orders, "Orders")
check_data_types(items, "Order Items")
check_data_types(reviews, "Order Reviews")

Dataset: Orders
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object
Dataset: Order Items
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object
Dataset: Order Reviews
review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_ans

## Step 3: Handle Missing Values

In this step, missing values are investigated to understand their causes before deciding whether they should be removed, replaced, or retained.

In [10]:
from retailpulse.utils import analyze_missing_values

In [11]:
analyze_missing_values(orders, "Orders")
analyze_missing_values(products, "Products")
analyze_missing_values(reviews, "Order Reviews")
analyze_missing_values(items, "Order Items")
analyze_missing_values(payments, "Order Payments")
analyze_missing_values(sellers, "Sellers")
analyze_missing_values(categories, "Product Category Name Translation")
analyze_missing_values(geolocation, "Geolocation")
analyze_missing_values(customers, "Customers")

Dataset: Orders
                               Missing Values  Percentage (%)
order_delivered_customer_date            2965            2.98
order_delivered_carrier_date             1783            1.79
order_approved_at                         160            0.16
Dataset: Products
                            Missing Values  Percentage (%)
product_category_name                  610            1.85
product_name_lenght                    610            1.85
product_description_lenght             610            1.85
product_photos_qty                     610            1.85
product_weight_g                         2            0.01
product_length_cm                        2            0.01
product_height_cm                        2            0.01
product_width_cm                         2            0.01
Dataset: Order Reviews
                        Missing Values  Percentage (%)
review_comment_title             87656           88.34
review_comment_message           58247           58.70

### Observations

- Most datasets contain no missing values and are ready for analysis.
- Missing values in the Orders dataset are likely related to the order lifecycle (e.g., canceled or undelivered orders) and require further investigation before any cleaning action.
- The Products dataset contains missing values mainly in product-related attributes, suggesting that a small number of products have incomplete information.
- The Reviews dataset contains a high percentage of missing values in the review title and review message columns. This is expected because customers are not required to leave textual feedback when submitting a rating.
- No cleaning actions were applied at this stage because the causes of missing values must be understood before deciding how to handle them.

## Decision 1: Orders Dataset

In [12]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [13]:
orders.loc[orders["order_delivered_customer_date"].isna(),"order_status"]

6           invoiced
44           shipped
103         invoiced
128       processing
154          shipped
            ...     
99283       canceled
99313     processing
99347       canceled
99348    unavailable
99415    unavailable
Name: order_status, Length: 2965, dtype: str

### Decision: Orders Dataset

- Most missing values in the delivery-related columns are expected and correspond to orders that were canceled, unavailable, shipped, or still in progress.
- Therefore, these missing values represent the business process rather than data quality issues.
- Only a very small number of delivered orders (8 records) have a missing delivery date, which may indicate data inconsistencies and will be investigated later.
- No missing values were removed or imputed at this stage.

In [14]:
products[products["product_category_name"].isna()]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [15]:
products[
    products["product_category_name"].isna()
].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0


In [16]:
missing_products = products[
    products["product_category_name"].isna()
]

missing_products_orders = pd.merge(
    missing_products,
    items,
    on="product_id",
    how="inner"
)

missing_products_orders.shape

(1603, 15)

In [17]:
missing_products_orders["product_id"].nunique()

610

### Decision: Products Dataset

- All 610 products with missing descriptive information appear in the sales records.
- This was verified by merging the Products and Order Items datasets using `product_id`.
- Since these products were actually sold, removing them would result in losing valid business transactions.
- Therefore, these records will be retained, and the missing values will only be handled if needed for a specific analysis.

### Decision: Reviews Dataset

- The missing values in `review_comment_title` and `review_comment_message` are expected because customers are not required to leave written feedback.
- These missing values represent user behavior rather than data quality issues.
- Therefore, no imputation or deletion will be performed.
- The review score remains available for all records and can still be used for customer satisfaction analysis.

# Duplicate Values Analysis

In this section, we identify duplicate records across all datasets to determine whether they represent actual data quality issues or valid business records. Any duplicate records will be carefully investigated before deciding whether they should be removed.

In [18]:
from retailpulse.utils import check_duplicates

In [20]:
check_duplicates(orders, "Orders")
check_duplicates(products, "Products")
check_duplicates(customers, "Customers")
check_duplicates(items, "Order Items")
check_duplicates(payments, "Order Payments")
check_duplicates(reviews, "Order Reviews")
check_duplicates(sellers, "Sellers")
check_duplicates(categories, "Product Categories")
check_duplicates(geolocation, "Geolocation")

Dataset: Orders
Duplicate Rows: 0
Dataset: Products
Duplicate Rows: 0
Dataset: Customers
Duplicate Rows: 0
Dataset: Order Items
Duplicate Rows: 0
Dataset: Order Payments
Duplicate Rows: 0
Dataset: Order Reviews
Duplicate Rows: 0
Dataset: Sellers
Duplicate Rows: 0
Dataset: Product Categories
Duplicate Rows: 0
Dataset: Geolocation
Duplicate Rows: 261831


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
15,1046,-23.546081,-46.644820,sao paulo,SP
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
66,1009,-23.546935,-46.636588,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP


In [ ]:
geolocation.drop_duplicates().shape

(738332, 5)

In [23]:
geolocation.shape

(1000163, 5)

In [26]:
geolocation = geolocation.drop_duplicates()

In [27]:
check_duplicates(geolocation, "Geolocation")

Dataset: Geolocation
Duplicate Rows: 0


In [28]:
print(geolocation.shape)

(738332, 5)


### Observation

- The Geolocation dataset contained 261,831 fully duplicated rows.
- Inspection confirmed that these rows were identical across all columns.
- Since they did not provide any additional information, duplicate records were removed using `drop_duplicates()`.
- After cleaning, no duplicate rows remain in the dataset.